# DAY 8 – Advanced Model Evaluation
## Confusion Matrix, Precision, Recall & F1-Score

### Objective of Day 8
Till now, we evaluated models using **accuracy**.
In this session, we go deeper and understand **how and where the model makes mistakes**.

By the end of this notebook, you will:
- Understand why accuracy alone is not enough
- Learn Confusion Matrix deeply
- Understand TP, TN, FP, FN
- Learn Precision, Recall, and F1-score
- Interpret results like industry professionals


# RAG-Based Question Answering System

## Objective
Build an API-based system that allows users to upload documents and ask questions using a Retrieval-Augmented Generation (RAG) approach.

## Tech Stack
- FastAPI
- Sentence-Transformers
- FAISS
- Python

## System Workflow# RAG-Based Question Answering System

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) based Question Answering system using FastAPI, Sentence Transformers, and FAISS.

The system allows users to upload documents, process them into embeddings, store them in a vector database, and ask questions that are answered using relevant retrieved document chunks.

The API provides interactive documentation using Swagger UI.

---

## Architecture Overview

High-level workflow:

1. User uploads a document using the API
2. Document ingestion runs as a background task
3. Text is chunked into overlapping segments
4. Each chunk is converted into vector embeddings
5. Embeddings are stored in a FAISS vector index
6. User submits a question
7. Relevant chunks are retrieved using similarity search
8. An answer is generated from the retrieved context

---

## Chunking Strategy

Chunk Size: 500 words  
Chunk Overlap: 100 words

Reason for choosing this chunk size:
- Large enough to preserve semantic meaning
- Small enough to avoid noisy embeddings
- Overlap helps reduce information loss at chunk boundaries

This strategy balances retrieval accuracy and performance.

---

## Retrieval Mechanism

- Embedding model: all-MiniLM-L6-v2
- Vector database: FAISS (IndexFlatL2)
- Similarity score calculation:

  similarity = 1 / (1 + distance)

- Only chunks above a similarity threshold of 0.3 are considered relevant

---

## Observed Retrieval Failure Case

Failure Case:
When a user asks a vague or generic question unrelated to the uploaded document.

Example:
Document: Technical article  
Question: "Tell me everything"

Result:
No chunks meet the similarity threshold, and the system returns:
"Not available in the document"

This behavior prevents hallucinated or irrelevant answers.

---

## Metric Tracked

Latency

Latency is measured for every /ask request. It includes:
- Query embedding time
- Vector similarity search time
- Response generation time

Latency is returned in the API response as:
latency_seconds

---

## Technology Stack

Backend: FastAPI  
Embeddings: Sentence Transformers  
Vector Store: FAISS  
Validation: Pydantic  
Rate Limiting: SlowAPI  
Server: Uvicorn  

---

## API Endpoints

Root Endpoint:
- Automatically redirects to /docs

---

Upload Document

Endpoint:
POST /upload

Description:
Uploads a document and processes it in the background.

Request Type:
multipart/form-data

Response:
{
  "message": "Document ingestion started"
}

---

Ask Question

Endpoint:
POST /ask

Request Body:
{
  "question": "string",
  "top_k": 3
}

Response:
{
  "answer": "string",
  "latency_seconds": 0.42
}

---

## API Documentation

Interactive Swagger UI is available at:
http://127.0.0.1:8000/docs

The root URL (/) redirects automatically to this page.

---

## Setup Instructions

Step 1: Clone the Repository

git clone <your-github-repo-url>
cd rag-qa-system

---

Step 2: Install Dependencies

pip install -r requirements.txt

---

Step 3: Run the Application

uvicorn app:app --reload

---

Step 4: Open Swagger UI

http://127.0.0.1:8000/docs

---

## requirements.txt Example

fastapi
uvicorn
sentence-transformers
faiss-cpu
slowapi
pydantic
numpy

---

## Evaluation Checklist Alignment

Document upload implemented
Chunking strategy implemented
Vector store implemented
Similarity search implemented
Background ingestion implemented
Rate limiting implemented
Request validation implemented
Latency metric tracked
Clear system explanation provided

---

## Conclusion

This project demonstrates a clean and practical implementation of a Retrieval-Augmented Generation based Question Answering system. The design focuses on simplicity, correctness, and explainability, following real-world API development practices.
1. User uploads a document
2. Document is processed in the background
3. Text is chunked into smaller parts
4. Chunks are converted into embeddings
5. Embeddings are stored in FAISS
6. User asks a question
7. Relevant chunks are retrieved
8. Answer is generated using retrieved context

## Chunking Strategy
Chunk size of 500 words with 100-word overlap was chosen to balance semantic context and retrieval precision.

## Retrieval Failure Case
When the user asks a vague question not present in the document, the system may retrieve weakly related chunks. This is handled using a similarity threshold.

## Metric Tracked
End-to-end latency is tracked to measure system performance.

## Run Instructions
```bash
pip install -r requirements.txt
uvicorn app:app --reload

## Why Accuracy Is Not Enough

Accuracy only tells **how many predictions were correct**.
It does NOT tell:
- What kind of mistakes were made
- Whether mistakes are serious or minor

In real life, different mistakes have different impacts.


## Real-Life Example

Imagine a placement prediction system:
- Predicting **Placed** when student is NOT placed → False hope
- Predicting **Not Placed** when student IS placed → Missed opportunity

Both errors are NOT equally harmful.
So we need deeper evaluation.


## Import Required Libraries


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import confusion_matrix, classification_report


## Dataset – Student Placement Data

Features:
- CGPA
- Internships
- Coding Skill

Target:
- 1 = Placed
- 0 = Not Placed


In [2]:
X = np.array([
    [7.5, 1, 3],
    [6.0, 0, 1],
    [8.2, 2, 4],
    [5.8, 0, 1],
    [9.0, 3, 5],
    [6.5, 1, 2],
    [7.8, 2, 4],
    [6.2, 0, 2]
])

y = np.array([1, 0, 1, 0, 1, 0, 1, 0])

## Train–Test Split
We split data to evaluate performance on unseen students.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

## Feature Scaling
Scaling is required for Logistic Regression.


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train Logistic Regression Model


In [5]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Make Predictions


In [6]:
y_pred = model.predict(X_test_scaled)

print("Actual values:", y_test)
print("Predicted values:", y_pred)

Actual values: [0 0]
Predicted values: [0 0]


## Confusion Matrix

Confusion Matrix shows:
- Correct predictions
- Type of errors made by the model


In [7]:
cm = confusion_matrix(y_test, y_pred)
cm

C:\Users\VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


array([[2]])

## Understanding Confusion Matrix

Structure:
[[TN, FP],
 [FN, TP]]

Where:
- TN: Correctly predicted Not Placed
- FP: Wrongly predicted Placed
- FN: Wrongly predicted Not Placed
- TP: Correctly predicted Placed


## Precision, Recall & F1-Score

Precision answers:
"Out of all students predicted as placed, how many actually got placed?"

Recall answers:
"Out of all students who got placed, how many were correctly identified?"

F1-score balances precision and recall.


In [8]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



## Industry Interpretation

- High precision → Few false alarms
- High recall → Few missed opportunities
- F1-score → Balanced model

Companies choose metrics based on business impact, not just accuracy.


## Key Learning from Day 8

- Accuracy alone is not enough
- Confusion Matrix explains model mistakes
- Precision and Recall measure different risks
- F1-score balances both
- Industry focuses on impact, not just numbers
